In [1]:
import math
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# EXACT CONFIGURATION FROM TRAINING
# ============================================================

IMG_SIZE = 32
IN_CHANNELS = 3
NUM_CLASSES = 10

PATCHES_PER_SIDE = 8
PATCH_SIZE = IMG_SIZE // PATCHES_PER_SIDE

NUM_PATCHES = PATCHES_PER_SIDE ** 2
NUM_TOKENS = NUM_PATCHES + 1

HIDDEN = 384
HEADS = 12
NUM_LAYERS = 7
MLP_HIDDEN = 384
DROPOUT = 0.0

print("========== MODEL CONFIGURATION ==========")
print(f"Image size       : {IMG_SIZE} x {IMG_SIZE}")
print(f"Input channels   : {IN_CHANNELS}")
print(f"Patches/side     : {PATCHES_PER_SIDE}")
print(f"Patch size       : {PATCH_SIZE} x {PATCH_SIZE}")
print(f"Number of patches: {NUM_PATCHES}")
print(f"CLS token        : Yes")
print(f"Total tokens     : {NUM_TOKENS}")
print(f"Hidden dimension : {HIDDEN}")
print(f"Attention heads  : {HEADS}")
print(f"Transformer depth: {NUM_LAYERS}")
print(f"MLP hidden       : {MLP_HIDDEN}")
print(f"Classes          : {NUM_CLASSES}")
print("==========================================")

========== MODEL CONFIGURATION ==========
Image size       : 32 x 32
Input channels   : 3
Patches/side     : 8
Patch size       : 4 x 4
Number of patches: 64
CLS token        : Yes
Total tokens     : 65
Hidden dimension : 384
Attention heads  : 12
Transformer depth: 7
MLP hidden       : 384
Classes          : 10


In [2]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, feats, head=8, dropout=0.0):
        super().__init__()

        self.head = head
        self.feats = feats
        self.sqrt_d = feats ** 0.5

        self.q = nn.Linear(feats, feats)
        self.k = nn.Linear(feats, feats)
        self.v = nn.Linear(feats, feats)

        self.o = nn.Linear(feats, feats)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        b, n, f = x.size()

        q = self.q(x).view(
            b, n, self.head, self.feats // self.head
        ).transpose(1, 2)

        k = self.k(x).view(
            b, n, self.head, self.feats // self.head
        ).transpose(1, 2)

        v = self.v(x).view(
            b, n, self.head, self.feats // self.head
        ).transpose(1, 2)

        score = F.softmax(
            torch.einsum(
                "bhif,bhjf->bhij",
                q,
                k
            ) / self.sqrt_d,
            dim=-1
        )

        attn = torch.einsum(
            "bhij,bhjf->bihf",
            score,
            v
        )

        o = self.dropout(
            self.o(attn.flatten(2))
        )

        return o


class TransformerEncoder(nn.Module):

    def __init__(
        self,
        feats,
        mlp_hidden,
        head=8,
        dropout=0.0
    ):
        super().__init__()

        self.la1 = nn.LayerNorm(feats)

        self.msa = MultiHeadSelfAttention(
            feats,
            head=head,
            dropout=dropout
        )

        self.la2 = nn.LayerNorm(feats)

        self.mlp = nn.Sequential(
            nn.Linear(feats, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, feats),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):

        out = self.msa(self.la1(x)) + x

        out = self.mlp(self.la2(out)) + out

        return out


class ViT(nn.Module):

    def __init__(
        self,
        in_c=3,
        num_classes=10,
        img_size=32,
        patch=8,
        dropout=0.0,
        num_layers=7,
        hidden=384,
        mlp_hidden=384,
        head=12,
        is_cls_token=True
    ):

        super().__init__()

        self.patch = patch
        self.is_cls_token = is_cls_token

        self.patch_size = img_size // patch

        f = (
            (img_size // patch) ** 2
            * in_c
        )

        num_tokens = (
            patch ** 2 + 1
            if is_cls_token
            else patch ** 2
        )

        self.emb = nn.Linear(
            f,
            hidden
        )

        self.cls_token = (
            nn.Parameter(
                torch.randn(1, 1, hidden)
            )
            if is_cls_token
            else None
        )

        self.pos_emb = nn.Parameter(
            torch.randn(
                1,
                num_tokens,
                hidden
            )
        )

        enc_list = [
            TransformerEncoder(
                hidden,
                mlp_hidden=mlp_hidden,
                dropout=dropout,
                head=head
            )
            for _ in range(num_layers)
        ]

        self.enc = nn.Sequential(*enc_list)

        self.fc = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x):

        out = self._to_words(x)

        out = self.emb(out)

        if self.is_cls_token:

            out = torch.cat(
                [
                    self.cls_token.repeat(
                        out.size(0), 1, 1
                    ),
                    out
                ],
                dim=1
            )

        out = out + self.pos_emb

        out = self.enc(out)

        if self.is_cls_token:
            out = out[:, 0]
        else:
            out = out.mean(1)

        out = self.fc(out)

        return out

    def _to_words(self, x):

        out = (
            x.unfold(
                2,
                self.patch_size,
                self.patch_size
            )
            .unfold(
                3,
                self.patch_size,
                self.patch_size
            )
            .permute(
                0, 2, 3, 4, 5, 1
            )
        )

        out = out.reshape(
            x.size(0),
            self.patch ** 2,
            -1
        )

        return out

In [3]:
model = ViT(
    in_c=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE,
    patch=PATCHES_PER_SIDE,
    dropout=DROPOUT,
    num_layers=NUM_LAYERS,
    hidden=HIDDEN,
    mlp_hidden=MLP_HIDDEN,
    head=HEADS,
    is_cls_token=True
)

model.eval()

print(model)

ViT(
  (emb): Linear(in_features=48, out_features=384, bias=True)
  (enc): Sequential(
    (0): TransformerEncoder(
      (la1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (msa): MultiHeadSelfAttention(
        (q): Linear(in_features=384, out_features=384, bias=True)
        (k): Linear(in_features=384, out_features=384, bias=True)
        (v): Linear(in_features=384, out_features=384, bias=True)
        (o): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (la2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=384, out_features=384, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=384, out_features=384, bias=True)
        (4): GELU(approximate='none')
        (5): Dropout(p=0.0, inplace=False)
      )
    )
    (1): TransformerEncoder(
      (la1): LayerNorm((384,),

In [4]:
dummy = torch.randn(1, 3, 32, 32)

with torch.no_grad():
    output = model(dummy)

print("Input :", dummy.shape)
print("Output:", output.shape)

Input : torch.Size([1, 3, 32, 32])
Output: torch.Size([1, 10])


In [5]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Parameters (M)       : {total_params / 1e6:.4f} M")

Total parameters     : 6,268,810
Trainable parameters : 6,268,810
Parameters (M)       : 6.2688 M


In [6]:
fp32_size = total_params * 4
int8_size = total_params * 1

print(f"FP32 weight size : {fp32_size / 1024:.2f} KB")
print(f"FP32 weight size : {fp32_size / 1024**2:.4f} MB")

print(f"INT8 weight size : {int8_size / 1024:.2f} KB")
print(f"INT8 weight size : {int8_size / 1024**2:.4f} MB")

FP32 weight size : 24487.54 KB
FP32 weight size : 23.9136 MB
INT8 weight size : 6121.88 KB
INT8 weight size : 5.9784 MB


In [7]:
# ============================================================
# FLOPs — EXACT ViT-CIFAR IMPLEMENTATION
# ============================================================

def calculate_vit_flops(
    img_size=32,
    in_channels=3,
    patches_per_side=8,
    hidden=384,
    heads=12,
    num_layers=7,
    mlp_hidden=384,
    num_classes=10
):
    """
    FLOP estimate for the exact ViT implementation used
    in our training notebook.

    Convention:
        One multiply + one addition = 2 FLOPs.

    Batch size = 1.
    """

    num_patches = patches_per_side ** 2
    num_tokens = num_patches + 1

    patch_size = img_size // patches_per_side
    patch_vector = patch_size * patch_size * in_channels

    head_dim = hidden // heads

    breakdown = {}

    # --------------------------------------------------------
    # Patch extraction
    # --------------------------------------------------------
    # _to_words() mainly performs reshape/unfold operations.
    # These are memory operations, not arithmetic FLOPs.
    breakdown["Patch extraction"] = 0

    # --------------------------------------------------------
    # Patch embedding
    # Linear:
    # patch_vector -> hidden
    # --------------------------------------------------------
    patch_embedding = (
        2
        * num_patches
        * patch_vector
        * hidden
    )

    breakdown["Patch embedding"] = patch_embedding

    total_flops = patch_embedding

    # --------------------------------------------------------
    # Transformer blocks
    # --------------------------------------------------------

    for layer in range(num_layers):

        layer_flops = 0

        # ---- LayerNorm 1 ----
        # Approximate arithmetic operations
        layernorm1 = 10 * num_tokens * hidden

        layer_flops += layernorm1

        # ---- Q projection ----
        q_flops = 2 * num_tokens * hidden * hidden

        # ---- K projection ----
        k_flops = 2 * num_tokens * hidden * hidden

        # ---- V projection ----
        v_flops = 2 * num_tokens * hidden * hidden

        layer_flops += q_flops + k_flops + v_flops

        # ---- QK^T ----
        qk_flops = (
            2
            * heads
            * num_tokens
            * num_tokens
            * head_dim
        )

        layer_flops += qk_flops

        # ---- Softmax ----
        # Approximate arithmetic cost.
        # Exact cost depends on implementation.
        softmax_flops = (
            5
            * heads
            * num_tokens
            * num_tokens
        )

        layer_flops += softmax_flops

        # ---- Attention × V ----
        av_flops = (
            2
            * heads
            * num_tokens
            * num_tokens
            * head_dim
        )

        layer_flops += av_flops

        # ---- Output projection ----
        output_projection = (
            2
            * num_tokens
            * hidden
            * hidden
        )

        layer_flops += output_projection

        # ---- LayerNorm 2 ----
        layernorm2 = 10 * num_tokens * hidden

        layer_flops += layernorm2

        # ---- MLP Linear 1 ----
        mlp_fc1 = (
            2
            * num_tokens
            * hidden
            * mlp_hidden
        )

        # ---- MLP Linear 2 ----
        mlp_fc2 = (
            2
            * num_tokens
            * mlp_hidden
            * hidden
        )

        layer_flops += mlp_fc1 + mlp_fc2

        # ---- GELU ----
        # Approximate as element-wise cost.
        # We have TWO GELUs in the actual implementation.
        gelu_flops = (
            20
            * num_tokens
            * mlp_hidden
            * 2
        )

        layer_flops += gelu_flops

        breakdown[f"Transformer block {layer + 1}"] = layer_flops

        total_flops += layer_flops

    # --------------------------------------------------------
    # Final LayerNorm
    # --------------------------------------------------------

    final_layernorm = 10 * hidden
    breakdown["Final LayerNorm"] = final_layernorm

    total_flops += final_layernorm

    # --------------------------------------------------------
    # Classifier
    # --------------------------------------------------------

    classifier = (
        2
        * hidden
        * num_classes
    )

    breakdown["Classifier"] = classifier

    total_flops += classifier

    return total_flops, breakdown


total_flops, flop_breakdown = calculate_vit_flops(
    img_size=IMG_SIZE,
    in_channels=IN_CHANNELS,
    patches_per_side=PATCHES_PER_SIDE,
    hidden=HIDDEN,
    heads=HEADS,
    num_layers=NUM_LAYERS,
    mlp_hidden=MLP_HIDDEN,
    num_classes=NUM_CLASSES
)

print("=" * 60)
print("FLOPs ANALYSIS")
print("=" * 60)

print(f"Total FLOPs/image : {total_flops:,}")
print(f"Total MFLOPs      : {total_flops / 1e6:.3f}")
print(f"Total GFLOPs      : {total_flops / 1e9:.6f}")

FLOPs ANALYSIS
Total FLOPs/image : 865,165,476
Total MFLOPs      : 865.165
Total GFLOPs      : 0.865165


In [8]:
# ============================================================
# FLOP BREAKDOWN
# ============================================================

flop_df = pd.DataFrame([
    {
        "Component": name,
        "FLOPs": value,
        "MFLOPs": value / 1e6,
        "Percentage": 100 * value / total_flops
    }
    for name, value in flop_breakdown.items()
])

display(flop_df)

,Component,FLOPs,MFLOPs,Percentage
0,Patch extraction,0,0.000000,0.000000
1,Patch embedding,2359296,2.359296,0.272699
2,Transformer block 1,123256380,123.256380,14.246567
3,Transformer block 2,123256380,123.256380,14.246567
4,Transformer block 3,123256380,123.256380,14.246567
5,Transformer block 4,123256380,123.256380,14.246567
6,Transformer block 5,123256380,123.256380,14.246567
7,Transformer block 6,123256380,123.256380,14.246567
8,Transformer block 7,123256380,123.256380,14.246567
9,Final LayerNorm,3840,0.003840,0.000444


In [9]:
# ============================================================
# ACTIVATION MEMORY — SINGLE IMAGE
# ============================================================

B = 1
N = NUM_TOKENS
E = HIDDEN
H = HEADS
D = E // H

FP32_BYTES = 4

print("=" * 60)
print("ACTIVATION MEMORY ANALYSIS")
print("=" * 60)

print(f"Batch size     : {B}")
print(f"Tokens         : {N}")
print(f"Embedding      : {E}")
print(f"Heads          : {H}")
print(f"Head dimension : {D}")

ACTIVATION MEMORY ANALYSIS
Batch size     : 1
Tokens         : 65
Embedding      : 384
Heads          : 12
Head dimension : 32


In [10]:
# ============================================================
# IMPORTANT ACTIVATION TENSORS
# ============================================================

activation_memory = {

    "Patch tokens":
        B * NUM_PATCHES * E,

    "Tokens + CLS":
        B * N * E,

    "Q":
        B * H * N * D,

    "K":
        B * H * N * D,

    "V":
        B * H * N * D,

    "Attention scores":
        B * H * N * N,

    "Attention probabilities":
        B * H * N * N,

    "Attention output":
        B * N * E,

    "MLP hidden":
        B * N * MLP_HIDDEN,
}

activation_rows = []

for name, elements in activation_memory.items():

    bytes_used = elements * FP32_BYTES

    activation_rows.append({
        "Tensor": name,
        "Elements": elements,
        "FP32 Bytes": bytes_used,
        "FP32 KB": bytes_used / 1024,
        "FP32 MB": bytes_used / (1024 ** 2)
    })

activation_df = pd.DataFrame(activation_rows)

display(activation_df)

,Tensor,Elements,FP32 Bytes,FP32 KB,FP32 MB
0,Patch tokens,24576,98304,96.000000,0.093750
1,Tokens + CLS,24960,99840,97.500000,0.095215
2,Q,24960,99840,97.500000,0.095215
3,K,24960,99840,97.500000,0.095215
4,V,24960,99840,97.500000,0.095215
5,Attention scores,50700,202800,198.046875,0.193405
6,Attention probabilities,50700,202800,198.046875,0.193405
7,Attention output,24960,99840,97.500000,0.095215
8,MLP hidden,24960,99840,97.500000,0.095215


In [11]:
# ============================================================
# PEAK LIVE MEMORY — CONSERVATIVE ESTIMATE
# ============================================================

token_memory = B * N * E * FP32_BYTES

q_memory = B * H * N * D * FP32_BYTES
k_memory = B * H * N * D * FP32_BYTES
v_memory = B * H * N * D * FP32_BYTES

attention_scores = (
    B * H * N * N * FP32_BYTES
)

attention_probs = (
    B * H * N * N * FP32_BYTES
)

attention_output = (
    B * N * E * FP32_BYTES
)

mlp_hidden_memory = (
    B * N * MLP_HIDDEN * FP32_BYTES
)

# Conservative attention working set
attention_peak = (
    token_memory
    + q_memory
    + k_memory
    + v_memory
    + attention_scores
    + attention_probs
    + attention_output
)

# Conservative MLP working set
mlp_peak = (
    token_memory
    + mlp_hidden_memory
    + token_memory
)

peak_activation_memory = max(
    attention_peak,
    mlp_peak
)

print("=" * 60)
print("PEAK ACTIVATION MEMORY")
print("=" * 60)

print(
    f"Attention working set : "
    f"{attention_peak / 1024:.2f} KB"
)

print(
    f"MLP working set       : "
    f"{mlp_peak / 1024:.2f} KB"
)

print(
    f"Estimated peak        : "
    f"{peak_activation_memory / 1024:.2f} KB"
)

print(
    f"Estimated peak        : "
    f"{peak_activation_memory / 1024**2:.4f} MB"
)

PEAK ACTIVATION MEMORY
Attention working set : 883.59 KB
MLP working set       : 292.50 KB
Estimated peak        : 883.59 KB
Estimated peak        : 0.8629 MB


In [12]:
# ============================================================
# FINAL EXP-00 BASELINE
# ============================================================

baseline_result = {
    "Experiment": "EXP-00",
    "Model": "ViT-CIFAR",
    "Dataset": "CIFAR-10",
    "Precision": "FP32",

    "Input_H": IMG_SIZE,
    "Input_W": IMG_SIZE,
    "Patch_Size": PATCH_SIZE,
    "Patches": NUM_PATCHES,
    "Tokens": NUM_TOKENS,

    "Embedding": HIDDEN,
    "Heads": HEADS,
    "Depth": NUM_LAYERS,
    "MLP_Hidden": MLP_HIDDEN,
    "Classes": NUM_CLASSES,

    "Parameters": total_params,
    "Parameters_M": total_params / 1e6,

    "FP32_Weight_MB": fp32_size / 1024**2,
    "INT8_Weight_MB": int8_size / 1024**2,

    "FLOPs": total_flops,
    "MFLOPs": total_flops / 1e6,

    "Peak_Activation_KB":
        peak_activation_memory / 1024,

    "Peak_Activation_MB":
        peak_activation_memory / 1024**2,
}

baseline_df = pd.DataFrame([baseline_result])

display(baseline_df)

os.makedirs("results", exist_ok=True)

baseline_df.to_csv(
    "results/baseline.csv",
    index=False
)

activation_df.to_csv(
    "results/activation_memory.csv",
    index=False
)

flop_df.to_csv(
    "results/flops.csv",
    index=False
)

print("\nSaved:")
print("results/baseline.csv")
print("results/activation_memory.csv")
print("results/flops.csv")

,Experiment,Model,Dataset,Precision,Input_H,Input_W,Patch_Size,Patches,Tokens,Embedding,...,MLP_Hidden,Classes,Parameters,Parameters_M,FP32_Weight_MB,INT8_Weight_MB,FLOPs,MFLOPs,Peak_Activation_KB,Peak_Activation_MB
0,EXP-00,ViT-CIFAR,CIFAR-10,FP32,32,32,4,64,65,384,...,384,10,6268810,6.26881,23.913612,5.978403,865165476,865.165476,883.59375,0.862885



Saved:
results/baseline.csv
results/activation_memory.csv
results/flops.csv


# Memory Aware Architecture Search

In [13]:
# ============================================================
# MEMORY-AWARE ARCHITECTURE SEARCH
# MCUFormer-ESP32 BTP
# ============================================================

import os
import math
import itertools
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Fixed project configuration
# ------------------------------------------------------------

IMG_SIZE = 32
IN_CHANNELS = 3
NUM_CLASSES = 10

# MCU-oriented design budget.
# This is a SEARCH CONSTRAINT, not a claim about total ESP32 SRAM.
SRAM_BUDGET_KB = 256.0

BYTES_PER_FP32 = 4
BYTES_PER_INT8 = 1

# ------------------------------------------------------------
# Search space
# ------------------------------------------------------------

PATCHES_PER_SIDE = [4, 8]
HIDDEN_DIMS = [64, 96, 128, 192, 256, 384]
HEADS = [2, 4, 8, 12]
DEPTHS = [2, 3, 4, 5]
MLP_RATIOS = [1, 2]

# ------------------------------------------------------------
# Parameter calculation
# Matches the actual ViT implementation used for training.
# ------------------------------------------------------------

def calculate_parameters(
    img_size,
    in_channels,
    patches_per_side,
    hidden,
    heads,
    depth,
    mlp_hidden,
    num_classes
):
    patch_size = img_size // patches_per_side
    patch_dim = patch_size * patch_size * in_channels
    num_patches = patches_per_side ** 2
    num_tokens = num_patches + 1

    total = 0

    # Patch embedding: Linear(patch_dim, hidden)
    total += patch_dim * hidden + hidden

    # CLS token
    total += hidden

    # Positional embedding
    total += num_tokens * hidden

    # Transformer blocks
    for _ in range(depth):

        # LayerNorm 1
        total += 2 * hidden

        # q, k, v, output projections
        # Each: Linear(hidden, hidden)
        total += 4 * (hidden * hidden + hidden)

        # LayerNorm 2
        total += 2 * hidden

        # MLP
        total += hidden * mlp_hidden + mlp_hidden
        total += mlp_hidden * hidden + hidden

    # Final LayerNorm
    total += 2 * hidden

    # Classifier
    total += hidden * num_classes + num_classes

    return total


# ------------------------------------------------------------
# FLOPs estimation
#
# Same convention used for the baseline:
# - multiply/add = 2 FLOPs
# - approximate LayerNorm
# - approximate GELU
# - approximate softmax
# ------------------------------------------------------------

def calculate_flops(
    img_size,
    in_channels,
    patches_per_side,
    hidden,
    heads,
    depth,
    mlp_hidden,
    num_classes
):
    patch_size = img_size // patches_per_side
    patch_dim = patch_size * patch_size * in_channels
    num_patches = patches_per_side ** 2
    num_tokens = num_patches + 1

    # Patch embedding
    patch_embed_flops = (
        2 * num_patches * patch_dim * hidden
    )

    # Per transformer block
    ln_flops = 10 * num_tokens * hidden

    # q, k, v projections
    qkv_flops = (
        3 * 2 * num_tokens * hidden * hidden
    )

    # Attention score QK^T
    attention_score_flops = (
        2 * heads * num_tokens * num_tokens * (hidden // heads)
    )

    # Softmax approximation
    softmax_flops = (
        5 * heads * num_tokens * num_tokens
    )

    # Attention x V
    attention_value_flops = (
        2 * heads * num_tokens * num_tokens * (hidden // heads)
    )

    # Output projection
    output_projection_flops = (
        2 * num_tokens * hidden * hidden
    )

    # MLP
    mlp_linear_flops = (
        2 * num_tokens * hidden * mlp_hidden
        + 2 * num_tokens * mlp_hidden * hidden
    )

    # Approximate GELU cost
    gelu_flops = (
        20 * num_tokens * mlp_hidden * 2
    )

    block_flops = (
        2 * ln_flops
        + qkv_flops
        + attention_score_flops
        + softmax_flops
        + attention_value_flops
        + output_projection_flops
        + mlp_linear_flops
        + gelu_flops
    )

    total_flops = (
        patch_embed_flops
        + depth * block_flops
        + (10 * num_tokens * hidden)       # final LN
        + (2 * hidden * num_classes)       # classifier
    )

    return int(total_flops)


# ------------------------------------------------------------
# Peak activation working-set estimation
#
# Conservative analytical estimate.
# This is NOT PyTorch allocated memory and NOT hardware SRAM.
# ------------------------------------------------------------

def calculate_activation_memory(
    patches_per_side,
    hidden,
    heads,
    mlp_hidden
):
    num_patches = patches_per_side ** 2
    num_tokens = num_patches + 1

    # Token activation
    tokens_bytes = num_tokens * hidden * BYTES_PER_FP32

    # Q/K/V
    q_bytes = tokens_bytes
    k_bytes = tokens_bytes
    v_bytes = tokens_bytes

    # Attention matrix
    attention_elements = (
        heads * num_tokens * num_tokens
    )

    attention_bytes = (
        attention_elements * BYTES_PER_FP32
    )

    # Attention probabilities
    probability_bytes = attention_bytes

    # Attention output
    attention_output_bytes = tokens_bytes

    # MLP hidden activation
    mlp_hidden_bytes = (
        num_tokens * mlp_hidden * BYTES_PER_FP32
    )

    # Conservative attention working set
    attention_peak_bytes = (
        tokens_bytes
        + q_bytes
        + k_bytes
        + v_bytes
        + attention_bytes
        + probability_bytes
        + attention_output_bytes
    )

    # Conservative MLP working set
    mlp_peak_bytes = (
        tokens_bytes
        + mlp_hidden_bytes
        + tokens_bytes
    )

    peak_bytes = max(
        attention_peak_bytes,
        mlp_peak_bytes
    )

    return {
        "tokens": num_tokens,
        "attention_peak_kb": attention_peak_bytes / 1024,
        "mlp_peak_kb": mlp_peak_bytes / 1024,
        "peak_activation_kb": peak_bytes / 1024,
        "peak_activation_mb": peak_bytes / (1024 ** 2),
    }


# ------------------------------------------------------------
# Generate candidate architectures
# ------------------------------------------------------------

candidates = []

candidate_id = 1

for patches_per_side in PATCHES_PER_SIDE:

    for hidden in HIDDEN_DIMS:

        for heads in HEADS:

            # Multi-head attention requires divisibility
            if hidden % heads != 0:
                continue

            for depth in DEPTHS:

                for mlp_ratio in MLP_RATIOS:

                    mlp_hidden = hidden * mlp_ratio

                    patch_size = IMG_SIZE // patches_per_side
                    num_patches = patches_per_side ** 2
                    num_tokens = num_patches + 1

                    params = calculate_parameters(
                        IMG_SIZE,
                        IN_CHANNELS,
                        patches_per_side,
                        hidden,
                        heads,
                        depth,
                        mlp_hidden,
                        NUM_CLASSES
                    )

                    flops = calculate_flops(
                        IMG_SIZE,
                        IN_CHANNELS,
                        patches_per_side,
                        hidden,
                        heads,
                        depth,
                        mlp_hidden,
                        NUM_CLASSES
                    )

                    memory = calculate_activation_memory(
                        patches_per_side,
                        hidden,
                        heads,
                        mlp_hidden
                    )

                    fp32_weight_mb = (
                        params * BYTES_PER_FP32
                        / (1024 ** 2)
                    )

                    int8_weight_mb = (
                        params * BYTES_PER_INT8
                        / (1024 ** 2)
                    )

                    feasible = (
                        memory["peak_activation_kb"]
                        <= SRAM_BUDGET_KB
                    )

                    candidates.append({
                        "candidate_id": f"C{candidate_id:03d}",
                        "patches_per_side": patches_per_side,
                        "patch_size": patch_size,
                        "num_patches": num_patches,
                        "tokens": num_tokens,
                        "hidden": hidden,
                        "heads": heads,
                        "depth": depth,
                        "mlp_ratio": mlp_ratio,
                        "mlp_hidden": mlp_hidden,
                        "parameters": params,
                        "parameters_M": params / 1e6,
                        "fp32_weight_MB": fp32_weight_mb,
                        "int8_weight_MB": int8_weight_mb,
                        "FLOPs": flops,
                        "MFLOPs": flops / 1e6,
                        "peak_activation_KB": memory["peak_activation_kb"],
                        "peak_activation_MB": memory["peak_activation_mb"],
                        "attention_peak_KB": memory["attention_peak_kb"],
                        "mlp_peak_KB": memory["mlp_peak_kb"],
                        "memory_feasible": feasible
                    })

                    candidate_id += 1


search_df = pd.DataFrame(candidates)

print("=" * 70)
print("MEMORY-AWARE ARCHITECTURE SEARCH")
print("=" * 70)

print(f"Total valid architectures : {len(search_df)}")
print(f"Activation memory budget  : {SRAM_BUDGET_KB:.1f} KB")

print(
    f"Memory-feasible candidates: "
    f"{search_df['memory_feasible'].sum()}"
)

print(
    f"Memory-infeasible         : "
    f"{(~search_df['memory_feasible']).sum()}"
)

print("=" * 70)


# ------------------------------------------------------------
# Add normalized resource metrics
# ------------------------------------------------------------

search_df["memory_utilization_%"] = (
    search_df["peak_activation_KB"]
    / SRAM_BUDGET_KB
    * 100
)

search_df["memory_margin_KB"] = (
    SRAM_BUDGET_KB
    - search_df["peak_activation_KB"]
)

# A simple capacity/resource indicator.
# Higher is better, but this is ONLY used to help shortlist.
search_df["capacity_density"] = (
    search_df["hidden"]
    * search_df["depth"]
    * search_df["mlp_ratio"]
    / search_df["MFLOPs"]
)


# ------------------------------------------------------------
# Save complete analytical search
# ------------------------------------------------------------

os.makedirs("results", exist_ok=True)

search_df.to_csv(
    "results/search_candidates.csv",
    index=False
)

print("\nSaved:")
print("results/search_candidates.csv")


# ------------------------------------------------------------
# Select a diverse shortlist
#
# We deliberately span the 256 KB budget instead of simply
# selecting the smallest models.
# ------------------------------------------------------------

feasible_df = search_df[
    search_df["memory_feasible"]
].copy()

# Memory targets across the usable budget
memory_targets = [
    64,
    96,
    128,
    160,
    192,
    224,
    240,
    256
]

selected_rows = []
remaining = feasible_df.copy()

for target in memory_targets:

    if len(remaining) == 0:
        break

    remaining = remaining.copy()

    # Distance from desired memory target
    remaining["target_distance"] = (
        abs(
            remaining["peak_activation_KB"]
            - target
        )
    )

    # Prefer lower compute when memory is similarly close
    remaining = remaining.sort_values(
        by=[
            "target_distance",
            "MFLOPs",
            "parameters"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )

    selected = remaining.iloc[0]

    selected_rows.append(selected)

    remaining = remaining[
        remaining["candidate_id"]
        != selected["candidate_id"]
    ]


selected_df = pd.DataFrame(selected_rows).copy()

# Sort by memory for readability
selected_df = selected_df.sort_values(
    "peak_activation_KB"
).reset_index(drop=True)

selected_df.to_csv(
    "results/search_selected.csv",
    index=False
)


# ------------------------------------------------------------
# Display shortlist
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SHORTLISTED ARCHITECTURES")
print("=" * 70)

display_columns = [
    "candidate_id",
    "patches_per_side",
    "patch_size",
    "tokens",
    "hidden",
    "heads",
    "depth",
    "mlp_ratio",
    "parameters_M",
    "int8_weight_MB",
    "MFLOPs",
    "peak_activation_KB",
    "memory_utilization_%"
]

print(
    selected_df[display_columns].to_string(
        index=False,
        formatters={
            "parameters_M": "{:.3f}".format,
            "int8_weight_MB": "{:.3f}".format,
            "MFLOPs": "{:.2f}".format,
            "peak_activation_KB": "{:.2f}".format,
            "memory_utilization_%": "{:.1f}".format
        }
    )
)

print("\nSaved:")
print("results/search_selected.csv")


# ------------------------------------------------------------
# Compare against EXP-00 baseline
# ------------------------------------------------------------

baseline = {
    "parameters_M": 6.26881,
    "fp32_weight_MB": 23.913612,
    "int8_weight_MB": 5.978403,
    "MFLOPs": 865.165476,
    "peak_activation_KB": 883.59375
}

print("\n" + "=" * 70)
print("BASELINE VS SEARCH SPACE")
print("=" * 70)

print(
    f"Baseline parameters       : "
    f"{baseline['parameters_M']:.3f} M"
)

print(
    f"Baseline FP32 weights     : "
    f"{baseline['fp32_weight_MB']:.3f} MB"
)

print(
    f"Baseline INT8 weights     : "
    f"{baseline['int8_weight_MB']:.3f} MB"
)

print(
    f"Baseline FLOPs            : "
    f"{baseline['MFLOPs']:.2f} MFLOPs"
)

print(
    f"Baseline peak activation  : "
    f"{baseline['peak_activation_KB']:.2f} KB"
)

print(
    f"\nSearch activation budget : "
    f"{SRAM_BUDGET_KB:.1f} KB"
)

print(
    f"Baseline / budget         : "
    f"{baseline['peak_activation_KB'] / SRAM_BUDGET_KB:.2f}x"
)

print("=" * 70)

MEMORY-AWARE ARCHITECTURE SEARCH
Total valid architectures : 336
Activation memory budget  : 256.0 KB
Memory-feasible candidates: 208
Memory-infeasible         : 128

Saved:
results/search_candidates.csv

SHORTLISTED ARCHITECTURES
candidate_id  patches_per_side  patch_size  tokens  hidden  heads  depth  mlp_ratio parameters_M int8_weight_MB MFLOPs peak_activation_KB memory_utilization_%
        C073                 4           8      17     128      8      2          1        0.228          0.217   8.08              60.56                 23.7
        C121                 4           8      17     256      4      2          1        0.849          0.809  29.49              94.03                 36.7
        C137                 4           8      17     384      2      2          1        1.863          1.776  64.27             132.02                 51.6
        C161                 4           8      17     384     12      2          1        1.863          1.776  64.30             15

# SHORTLISTED ARCHITECTURE TRAINING

In [ ]:
# ============================================================
# SHORTLISTED ARCHITECTURE TRAINING
# Memory-Aware ViT Search
# ============================================================

import os
import time
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SEED = 42

BATCH_SIZE = 128
EVAL_BATCH_SIZE = 1024

SEARCH_EPOCHS = 15

LR = 1e-3
WEIGHT_DECAY = 5e-5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# CIFAR-10 dataset
# Same basic training protocol as baseline
# ------------------------------------------------------------

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(
        policy=transforms.AutoAugmentPolicy.CIFAR10
    ),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])


train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training images :", len(train_dataset))
print("Test images     :", len(test_dataset))


# ------------------------------------------------------------
# Candidate architectures
#
# These are the architectures selected from the analytical
# memory search.
# ------------------------------------------------------------

CANDIDATES = [
    {
        "candidate_id": "C073",
        "patch": 4,
        "hidden": 128,
        "heads": 8,
        "depth": 2,
        "mlp_hidden": 128,
    },

    {
        "candidate_id": "C121",
        "patch": 4,
        "hidden": 256,
        "heads": 4,
        "depth": 2,
        "mlp_hidden": 256,
    },

    {
        "candidate_id": "C161",
        "patch": 4,
        "hidden": 384,
        "heads": 12,
        "depth": 2,
        "mlp_hidden": 384,
    },

    {
        "candidate_id": "C193",
        "patch": 8,
        "hidden": 96,
        "heads": 2,
        "depth": 2,
        "mlp_hidden": 96,
    },

    {
        "candidate_id": "C225",
        "patch": 8,
        "hidden": 128,
        "heads": 2,
        "depth": 2,
        "mlp_hidden": 128,
    },

    {
        "candidate_id": "C201",
        "patch": 8,
        "hidden": 96,
        "heads": 4,
        "depth": 2,
        "mlp_hidden": 96,
    },
]


# ------------------------------------------------------------
# Training / evaluation functions
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)


def train_one_epoch(
    model,
    optimizer,
    scaler
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        if DEVICE.type == "cuda":

            with torch.cuda.amp.autocast():

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()
            optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        running_loss / total,
        100.0 * correct / total
    )


@torch.no_grad()
def evaluate(model):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in test_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        running_loss / total,
        100.0 * correct / total
    )


# ------------------------------------------------------------
# Train candidates
# ------------------------------------------------------------

search_results = []

os.makedirs(
    "results/search_checkpoints",
    exist_ok=True
)

for candidate in CANDIDATES:

    cid = candidate["candidate_id"]

    print("\n")
    print("=" * 70)
    print(f"TRAINING CANDIDATE: {cid}")
    print("=" * 70)

    print(
        f"patch={candidate['patch']} | "
        f"hidden={candidate['hidden']} | "
        f"heads={candidate['heads']} | "
        f"depth={candidate['depth']} | "
        f"mlp={candidate['mlp_hidden']}"
    )

    model = ViT(
        in_c=3,
        num_classes=10,
        img_size=32,
        patch=candidate["patch"],
        dropout=0.0,
        num_layers=candidate["depth"],
        hidden=candidate["hidden"],
        mlp_hidden=candidate["mlp_hidden"],
        head=candidate["heads"],
        is_cls_token=True
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        betas=(0.9, 0.999),
        weight_decay=WEIGHT_DECAY
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

    best_acc = 0.0
    best_state = None

    start_time = time.time()

    for epoch in range(1, SEARCH_EPOCHS + 1):

        train_loss, train_acc = train_one_epoch(
            model,
            optimizer,
            scaler
        )

        val_loss, val_acc = evaluate(
            model
        )

        if val_acc > best_acc:

            best_acc = val_acc

            best_state = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"[{cid}] "
            f"Epoch {epoch:02d}/{SEARCH_EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.2f}% | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.2f}%"
        )

    elapsed = time.time() - start_time

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    # --------------------------------------------------------
    # Resource metrics
    # --------------------------------------------------------

    params = sum(
        p.numel()
        for p in model.parameters()
    )

    fp32_mb = (
        params * 4
        / (1024 ** 2)
    )

    int8_mb = (
        params
        / (1024 ** 2)
    )

    # Reuse analytical functions from the previous section
    memory = calculate_activation_memory(
        candidate["patch"],
        candidate["hidden"],
        candidate["heads"],
        candidate["mlp_hidden"]
    )

    flops = calculate_flops(
        IMG_SIZE,
        IN_CHANNELS,
        candidate["patch"],
        candidate["hidden"],
        candidate["heads"],
        candidate["depth"],
        candidate["mlp_hidden"],
        NUM_CLASSES
    )

    # Save best checkpoint
    checkpoint_path = (
        f"results/search_checkpoints/"
        f"{cid}.pth"
    )

    torch.save(
        {
            "candidate_id": cid,
            "model_state_dict": model.state_dict(),
            "best_val_accuracy": best_acc,
            "architecture": candidate
        },
        checkpoint_path
    )

    search_results.append({

        "candidate_id": cid,

        "patches_per_side":
            candidate["patch"],

        "patch_size":
            IMG_SIZE // candidate["patch"],

        "tokens":
            candidate["patch"] ** 2 + 1,

        "hidden":
            candidate["hidden"],

        "heads":
            candidate["heads"],

        "depth":
            candidate["depth"],

        "mlp_hidden":
            candidate["mlp_hidden"],

        "parameters":
            params,

        "parameters_M":
            params / 1e6,

        "FP32_weight_MB":
            fp32_mb,

        "INT8_weight_MB":
            int8_mb,

        "MFLOPs":
            flops / 1e6,

        "peak_activation_KB":
            memory["peak_activation_kb"],

        "best_val_accuracy":
            best_acc,

        "training_time_min":
            elapsed / 60.0,

        "checkpoint":
            checkpoint_path
    })

    # Free GPU memory
    del model
    del optimizer
    del scaler

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Final search results
# ------------------------------------------------------------

results_df = pd.DataFrame(
    search_results
)

results_df = results_df.sort_values(
    by="best_val_accuracy",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# Add memory efficiency
# ------------------------------------------------------------

results_df["memory_utilization_%"] = (
    results_df["peak_activation_KB"]
    / SRAM_BUDGET_KB
    * 100
)

results_df["memory_margin_KB"] = (
    SRAM_BUDGET_KB
    - results_df["peak_activation_KB"]
)

results_df["memory_feasible"] = (
    results_df["peak_activation_KB"]
    <= SRAM_BUDGET_KB
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results_df.to_csv(
    "results/search_results.csv",
    index=False
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n")
print("=" * 90)
print("FINAL MEMORY-AWARE SEARCH RESULTS")
print("=" * 90)

display_columns = [
    "candidate_id",
    "tokens",
    "hidden",
    "heads",
    "depth",
    "mlp_hidden",
    "parameters_M",
    "INT8_weight_MB",
    "MFLOPs",
    "peak_activation_KB",
    "best_val_accuracy",
    "memory_feasible"
]

print(
    results_df[
        display_columns
    ].to_string(
        index=False,
        formatters={
            "parameters_M":
                "{:.3f}".format,

            "INT8_weight_MB":
                "{:.3f}".format,

            "MFLOPs":
                "{:.2f}".format,

            "peak_activation_KB":
                "{:.2f}".format,

            "best_val_accuracy":
                "{:.2f}".format
        }
    )
)

print("\nSaved:")
print("results/search_results.csv")

print("\n" + "=" * 90)
print("MEMORY-FEASIBLE MODELS")
print("=" * 90)

feasible_results = results_df[
    results_df["memory_feasible"]
]

print(
    feasible_results[
        display_columns
    ].to_string(
        index=False,
        formatters={
            "parameters_M":
                "{:.3f}".format,

            "INT8_weight_MB":
                "{:.3f}".format,

            "MFLOPs":
                "{:.2f}".format,

            "peak_activation_KB":
                "{:.2f}".format,

            "best_val_accuracy":
                "{:.2f}".format
        }
    )
)

Device: cuda
GPU: Tesla T4


100%|██████████| 170M/170M [40:14<00:00, 70.6kB/s] 


Training images : 50000
Test images     : 10000


TRAINING CANDIDATE: C073
patch=4 | hidden=128 | heads=8 | depth=2 | mlp=128


/tmp/ipykernel_58/4193154166.py:356: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
/tmp/ipykernel_58/4193154166.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[C073] Epoch 01/15 | train_loss=2.0851 | train_acc=25.10% | val_loss=1.7933 | val_acc=40.93%
[C073] Epoch 02/15 | train_loss=1.9440 | train_acc=32.69% | val_loss=1.7406 | val_acc=42.93%
[C073] Epoch 03/15 | train_loss=1.8905 | train_acc=34.91% | val_loss=1.6821 | val_acc=45.76%
[C073] Epoch 04/15 | train_loss=1.8504 | train_acc=36.63% | val_loss=1.6537 | val_acc=46.33%
[C073] Epoch 05/15 | train_loss=1.8239 | train_acc=38.04% | val_loss=1.5935 | val_acc=50.16%
[C073] Epoch 06/15 | train_loss=1.7905 | train_acc=39.90% | val_loss=1.6198 | val_acc=48.88%
[C073] Epoch 07/15 | train_loss=1.7701 | train_acc=40.96% | val_loss=1.6153 | val_acc=48.50%
[C073] Epoch 08/15 | train_loss=1.7511 | train_acc=41.85% | val_loss=1.5858 | val_acc=50.41%
[C073] Epoch 09/15 | train_loss=1.7328 | train_acc=42.84% | val_loss=1.5398 | val_acc=51.99%
[C073] Epoch 10/15 | train_loss=1.7116 | train_acc=44.07% | val_loss=1.5304 | val_acc=52.37%
[C073] Epoch 11/15 | train_loss=1.6946 | train_acc=44.82% | val_loss=1

In [17]:
import os

for cid in ["C073", "C121", "C161", "C193", "C225", "C201"]:
    path = f"results/search_checkpoints/{cid}.pth"
    print(cid, os.path.exists(path))

C073 True
C121 True
C161 True
C193 True
C225 True
C201 True


In [18]:
import os
import torch

for cid in ["C073", "C121", "C161", "C193", "C225", "C201"]:

    path = f"results/search_checkpoints/{cid}.pth"

    ckpt = torch.load(
        path,
        map_location="cpu"
    )

    print(
        f"{cid}: "
        f"best_val_acc={ckpt['best_val_accuracy']:.2f}%"
    )

C073: best_val_acc=56.28%
C121: best_val_acc=56.74%
C161: best_val_acc=58.27%
C193: best_val_acc=57.32%
C225: best_val_acc=60.01%
C201: best_val_acc=60.34%


In [19]:
checkpoint = torch.load(
    "results/search_checkpoints/C201.pth",
    map_location="cpu"
)

print("Candidate:", checkpoint["candidate_id"])
print("Best validation accuracy:",
      checkpoint["best_val_accuracy"])
print("Architecture:")
print(checkpoint["architecture"])

Candidate: C201
Best validation accuracy: 60.34
Architecture:
{'candidate_id': 'C201', 'patch': 8, 'hidden': 96, 'heads': 4, 'depth': 2, 'mlp_hidden': 96}


In [20]:
import torch

ckpt = torch.load(
    "results/search_checkpoints/C201.pth",
    map_location="cpu"
)

print("Candidate:", ckpt["candidate_id"])
print("Best validation accuracy:", ckpt["best_val_accuracy"])
print("Architecture:", ckpt["architecture"])

Candidate: C201
Best validation accuracy: 60.34
Architecture: {'candidate_id': 'C201', 'patch': 8, 'hidden': 96, 'heads': 4, 'depth': 2, 'mlp_hidden': 96}
